In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import warnings

In [28]:
warnings.filterwarnings('ignore')

In [29]:
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv("../HouseTS.csv")

In [19]:
print("Columns:", df.columns.tolist())
print("\nFirst row:", df.iloc[0].to_dict())

Columns: ['date', 'median_sale_price', 'median_list_price', 'median_ppsf', 'median_list_ppsf', 'homes_sold', 'pending_sales', 'new_listings', 'inventory', 'median_dom', 'avg_sale_to_list', 'sold_above_list', 'off_market_in_two_weeks', 'city', 'zipcode', 'year', 'bank', 'bus', 'hospital', 'mall', 'park', 'restaurant', 'school', 'station', 'supermarket', 'Total Population', 'Median Age', 'Per Capita Income', 'Total Families Below Poverty', 'Total Housing Units', 'Median Rent', 'Median Home Value', 'Total Labor Force', 'Unemployed Population', 'Total School Age Population', 'Total School Enrollment', 'Median Commute Time', 'price', 'city_full']

First row: {'date': '2012-03-31', 'median_sale_price': 46550.0, 'median_list_price': 217450.0, 'median_ppsf': 31.813673766522705, 'median_list_ppsf': 110.18366606797456, 'homes_sold': 14.0, 'pending_sales': 23.0, 'new_listings': 44.0, 'inventory': 64.0, 'median_dom': 59.5, 'avg_sale_to_list': 0.9436623661734675, 'sold_above_list': 0.14285714285714

In [20]:
print("Dataset shape:", df.shape)
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print(f"Unique zipcodes: {df['zipcode'].nunique()}")
print(f"Unique cities: {df['city'].nunique()}")

Dataset shape: (884092, 39)
Date range: 2012-03-31 to 2023-12-31
Unique zipcodes: 6226
Unique cities: 30


In [21]:
# checking missing values
missing = df.isnull().sum()
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing, 'Percent': missing_percent})
missing_df[missing_df['Missing'] > 0].sort_values('Percent', ascending=False)
# no missing values yayyyy

,Missing,Percent


In [22]:
# converting the date column
df['date'] = pd.to_datetime(df['date'])

# adding time
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter

print("Date range:", df['date'].min(), "to", df['date'].max())

Date range: 2012-03-31 00:00:00 to 2023-12-31 00:00:00


In [23]:
# getting rid of price outliers 
# trying to keep price between 1st and 99th percentile
# dont rerun this after running once bc it'll just keep removing extreme vals
for col in ['median_sale_price', 'median_list_price', 'price']:
    if col in df.columns:
        lower = df[col].quantile(0.01)
        upper = df[col].quantile(0.99)
        df = df[(df[col] >= lower) & (df[col] <= upper)]
        print(f"{col}: kept prices between ${lower:,.0f} and ${upper:,.0f}")

print(f"\nShape after removing outliers: {df.shape}")

median_sale_price: kept prices between $22,500 and $1,787,500
median_list_price: kept prices between $45,000 and $1,549,000
price: kept prices between $61,239 and $1,230,234

Shape after removing outliers: (832411, 41)


In [24]:
# diff bt list price and the sale price
df['list_sale_gap'] = (df['median_list_price'] - df['median_sale_price']) / df['median_list_price']

# diff bt list price per square foot and sale price per square foot
df['ppsf_gap'] = (df['median_list_ppsf'] - df['median_ppsf']) / df['median_list_ppsf']

print("The new features I created are as follows:")
print("- list_sale_gap (how much below list price homes sell)")
print("- ppsf_gap (price per square foot gap)")


The new features I created are as follows:
- list_sale_gap (how much below list price homes sell)
- ppsf_gap (price per square foot gap)


In [25]:
import os

os.makedirs('../data/processed', exist_ok=True)
df.to_csv('../data/processed/housing_data_cleaned.csv', index=False)

print("Saved to: ../data/processed/housing_data_cleaned.csv")
print(f"Final shape: {df.shape}")

Saved to: ../data/processed/housing_data_cleaned.csv
Final shape: (832411, 43)


In [26]:
# checking dataframe integrity here
df[['date', 'zipcode', 'city', 'median_sale_price', 'list_sale_gap']].head(3)

print("\nPrice statistics:")
df['median_sale_price'].describe()


Price statistics:


count    8.324110e+05
mean     3.606066e+05
std      2.340703e+05
min      2.250000e+04
25%      1.910000e+05
50%      3.025000e+05
75%      4.650000e+05
max      1.787500e+06
Name: median_sale_price, dtype: float64